In [ ]:
import os
from pathlib import Path

REVIEW_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "REVIEW_CODE_ROOT").is_file()),
    None,
)
if REVIEW_ROOT is None:
    raise RuntimeError("Open this notebook from the extracted review package.")
os.chdir(REVIEW_ROOT)
REVIEW_FREQUENCY = os.environ.get("BFORTPROB_FREQUENCY", "daily")
if REVIEW_FREQUENCY not in {"daily", "weekly"}:
    raise ValueError("BFORTPROB_FREQUENCY must be daily or weekly.")


# Train/test split

The original chronological log-return series are first converted into overlapping supervised rolling windows using a lookback of 250 observations, a forecast horizon of 20 observations, and a slide of 1. The resulting window rows are then split chronologically: the first 80% form the training side and the final 20% form the test set. To prevent overlap between training and test target horizons, 19 training rows immediately preceding the test boundary are purged. The test set remains the complete final 20% of window rows.

In [ ]:
from pathlib import Path
import math

import numpy as np
import pandas as pd


# ============================================================
# SETTINGS
# ============================================================

LOOKBACK = 250
HORIZON = 20
SLIDE = 1
TRAIN_RATIO = 0.8

RAW_DIR = Path(f"data/chronological_log_returns/output_{REVIEW_FREQUENCY}")

SUPERVISED_DIR = Path(f"BForTFin_log_return/output_{REVIEW_FREQUENCY}")

SPLIT_DIR = Path(f"BForTFin_log_return/arima_split_datasets/output_{REVIEW_FREQUENCY}")

SUPERVISED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SPLIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

ASSET_CLASSES = [
    "G7",
    "EM",
    "MSCI",
    "Rates",
    "REIT",
    "Equities",
    "ETF",
    "Crypto",
]


# ============================================================
# HELPERS
# ============================================================

def make_supervised_dataset(
    values,
    lookback=LOOKBACK,
    horizon=HORIZON,
    slide=SLIDE,
):
    """
    Convert one chronological series into overlapping supervised
    rolling windows.

    Each row contains:

        X_0, ..., X_{lookback-1},
        Y_0, ..., Y_{horizon-1}

    where X is the observed context and Y is the future target.
    """

    values = np.asarray(
        values,
        dtype=float,
    ).reshape(-1)

    if not np.isfinite(values).all():
        raise ValueError(
            "Series contains NaN or infinite values."
        )

    if len(values) < lookback + horizon:
        raise ValueError(
            "Series is too short for the requested "
            "lookback and horizon."
        )

    rows = []

    for start in range(
        0,
        len(values) - lookback - horizon + 1,
        slide,
    ):

        x = values[
            start:
            start + lookback
        ]

        y = values[
            start + lookback:
            start + lookback + horizon
        ]

        rows.append(
            np.concatenate([
                x,
                y,
            ])
        )

    columns = (
        [
            f"X_{i}"
            for i in range(lookback)
        ]
        +
        [
            f"Y_{i}"
            for i in range(horizon)
        ]
    )

    return pd.DataFrame(
        rows,
        columns=columns,
    )


def extract_log_return_series(
    df,
    target_col=None,
):
    """
    Extract the chronological log-return series.

    If the input contains only one column, target_col can be omitted.
    """

    if target_col is None:

        if df.shape[1] != 1:
            raise ValueError(
                "target_col must be specified when the raw dataset "
                "contains more than one column."
            )

        values = df.iloc[:, 0]

    else:

        if target_col not in df.columns:
            raise ValueError(
                f"Column '{target_col}' was not found."
            )

        values = df[target_col]

    return values.to_numpy(
        dtype=float
    )


# ============================================================
# PURGE SIZE
# ============================================================

PURGE_ROWS = max(
    0,
    math.ceil(
        HORIZON / SLIDE
    ) - 1,
)

print(
    f"Purging {PURGE_ROWS} rows "
    f"at each train/test boundary."
)


# ============================================================
# PROCESS ALL DATASETS
# ============================================================

for asset_class in ASSET_CLASSES:

    raw_asset_dir = (
        RAW_DIR / asset_class
    )

    supervised_asset_dir = (
        SUPERVISED_DIR / asset_class
    )

    split_asset_dir = (
        SPLIT_DIR / asset_class
    )

    supervised_asset_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    split_asset_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    for csv_path in sorted(
        raw_asset_dir.glob("*.csv")
    ):

        # Keep Tether excluded
        if "tether" in csv_path.name.lower():
            continue

        raw_df = pd.read_csv(
            csv_path
        )

        if len(raw_df) == 0:
            print(
                f"Skipping empty dataset: "
                f"{csv_path.name}"
            )
            continue

        # ----------------------------------------------------
        # 1. Extract chronological log-return series
        # ----------------------------------------------------

        values = extract_log_return_series(
            raw_df,
            target_col=None,
        )

        # ----------------------------------------------------
        # 2. Convert chronological series to supervised
        #    rolling-window dataset
        # ----------------------------------------------------

        supervised_df = make_supervised_dataset(
            values,
            lookback=LOOKBACK,
            horizon=HORIZON,
            slide=SLIDE,
        )

        dataset_name = (
            csv_path.stem
        )

        # Optional: save the complete supervised dataset
        supervised_path = (
            supervised_asset_dir
            / (
                f"{dataset_name}"
                f"_no_preprocess"
                f"_lookback-{LOOKBACK}"
                f"_horizon-{HORIZON}"
                f"_slide-{SLIDE}.csv"
            )
        )

        supervised_df.to_csv(
            supervised_path,
            index=False,
        )

        # ----------------------------------------------------
        # 3. Chronological 80/20 split
        #
        # IMPORTANT:
        # split is over SUPERVISED WINDOW ROWS.
        # ----------------------------------------------------

        split_idx = int(
            len(supervised_df)
            * TRAIN_RATIO
        )

        # ----------------------------------------------------
        # 4. Purge training rows immediately before test
        # ----------------------------------------------------

        train_end = (
            split_idx
            - PURGE_ROWS
        )

        if train_end <= 0:
            print(
                f"Skipping {csv_path.name}: "
                f"not enough rows after purge."
            )
            continue

        train_df = supervised_df.iloc[
            :train_end
        ].copy()

        test_df = supervised_df.iloc[
            split_idx:
        ].copy()

        # ----------------------------------------------------
        # 5. Save train/test datasets
        # ----------------------------------------------------

        train_path = (
            split_asset_dir
            / (
                f"train_{dataset_name}"
                f"_no_preprocess"
                f"_lookback-{LOOKBACK}"
                f"_horizon-{HORIZON}"
                f"_slide-{SLIDE}.csv"
            )
        )

        test_path = (
            split_asset_dir
            / (
                f"test_{dataset_name}"
                f"_no_preprocess"
                f"_lookback-{LOOKBACK}"
                f"_horizon-{HORIZON}"
                f"_slide-{SLIDE}.csv"
            )
        )

        train_df.to_csv(
            train_path,
            index=False,
        )

        test_df.to_csv(
            test_path,
            index=False,
        )

        print(
            f"{asset_class} | "
            f"{dataset_name}: "
            f"{len(values)} chronological observations -> "
            f"{len(supervised_df)} supervised rows -> "
            f"{len(train_df)} train / "
            f"{PURGE_ROWS} purged / "
            f"{len(test_df)} test"
        )